# Enveda CASMI 2026 — FPNet + Spectrum Retrieval

This notebook combines two evidence channels: (1) a public pretrained MS/MS-to-fingerprint model, ranking mass-matched natural-product candidates by fingerprint likelihood; and (2) direct similarity to the competition’s reference spectra. It writes a validated `/kaggle/working/submission.csv` and does not require internet access.

**Sources and attribution:** FPNet checkpoints are from the public CC0 Kaggle dataset [CASMI26 fingerprint models](https://www.kaggle.com/datasets/prvsiyan/casmi26-fp-models-v2). Candidate structures and fingerprints are from [COCONUT 2.0 candidates](https://www.kaggle.com/datasets/prvsiyan/coconut-casmi26-candidates), licensed CC BY 4.0; see that dataset’s card for source attribution. This model combination has not yet been measured on the hidden test labels, so no leaderboard score is guaranteed. Run all cells, then submit the notebook’s completed version through Kaggle.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time

COMPETITION_SLUG = "enveda-CASMI26-molecule-id-mass-spectra"
INPUT_ROOT = Path("/kaggle/input")


def find_input(filename: str) -> Path:
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} under {INPUT_ROOT}. "
            "Attach the Enveda CASMI 2026 competition data to this notebook."
        )
    return matches[0]


TRAIN_PATH = find_input("train.parquet")
TEST_PATH = find_input("test.parquet")
SAMPLE_PATH = find_input("sample_submission.csv")

print("Train:", TRAIN_PATH)
print("Test:", TEST_PATH)
print("Sample:", SAMPLE_PATH)

# Read only the columns needed by this baseline. In particular, avoid
# materializing unused metadata columns from the large training parquet.
TRAIN_COLUMNS = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "precursor_mz",
    "ms2_mzs",
    "ms2_normalized_intensities",
]
train = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
test = pd.read_parquet(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH, dtype={"molecule_id": str})

required_test = {
    "molecule_id", "adduct", "precursor_mz",
    "ms2_mzs", "ms2_normalized_intensities",
}
missing = required_test - set(test.columns)
if missing:
    raise ValueError(f"Missing test columns: {sorted(missing)}")

train = train.dropna(
    subset=["inchikey14", "normalized_smiles", "adduct", "precursor_mz"]
).copy()
train["inchikey14"] = train["inchikey14"].astype(str)
train["normalized_smiles"] = train["normalized_smiles"].astype(str)
train["adduct"] = train["adduct"].astype(str)
train["precursor_mz"] = pd.to_numeric(train["precursor_mz"], errors="coerce")
train = train[
    np.isfinite(train["precursor_mz"])
    & train["normalized_smiles"].ne("")
    & train["inchikey14"].ne("")
].reset_index(drop=True)

test["molecule_id"] = test["molecule_id"].astype(str)
test["adduct"] = test["adduct"].fillna("").astype(str)
test["precursor_mz"] = pd.to_numeric(test["precursor_mz"], errors="coerce")

assert sample_submission.columns.tolist() == ["molecule_id", "smiles"]
assert sample_submission["molecule_id"].is_unique
assert test["molecule_id"].notna().all()

print(f"Training reference spectra: {len(train):,}")
print(f"Training reference molecules: {train['inchikey14'].nunique():,}")
print(f"Test spectra: {len(test):,}")
print(f"Test molecules: {test['molecule_id'].nunique():,}")
print(f"Sample rows: {len(sample_submission):,}")

In [ ]:
# Candidate retrieval and MS/MS fragment-similarity helpers.
# Spectra are converted to sparse 0.01 Da fragment bins and compared with
# normalized cosine similarity. Precursor windows widen only as needed.

FRAGMENT_BIN_DA = 0.01
PRECURSOR_WINDOWS_PPM = (2, 5, 10, 25)
MAX_CANDIDATE_SPECTRA = 2500
MAX_GUESSES = 25

# Sorted precursor-mass indexes make candidate lookup fast.
adduct_index = {}
for adduct, group in train.groupby("adduct", sort=False):
    positions = group.index.to_numpy(dtype=np.int64)
    masses = train.loc[positions, "precursor_mz"].to_numpy(dtype=np.float64)
    order = np.argsort(masses)
    adduct_index[str(adduct)] = (masses[order], positions[order])

all_masses = train["precursor_mz"].to_numpy(dtype=np.float64)
all_positions = np.arange(len(train), dtype=np.int64)
global_order = np.argsort(all_masses)
all_masses = all_masses[global_order]
all_positions = all_positions[global_order]


def spectrum_vector(mzs, intensities):
    """Return unique sorted fragment bins and unit-length sqrt intensities."""
    try:
        mzs = np.asarray(mzs, dtype=np.float64).reshape(-1)
        intensities = np.asarray(intensities, dtype=np.float64).reshape(-1)
    except (TypeError, ValueError):
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)

    if len(mzs) == 0 or len(mzs) != len(intensities):
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)

    valid = np.isfinite(mzs) & np.isfinite(intensities) & (mzs >= 0) & (intensities > 0)
    mzs, intensities = mzs[valid], intensities[valid]
    if len(mzs) == 0:
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)

    bins = np.rint(mzs / FRAGMENT_BIN_DA).astype(np.int32)
    unique_bins, inverse = np.unique(bins, return_inverse=True)
    values = np.bincount(inverse, weights=np.sqrt(intensities)).astype(np.float32)
    norm = np.linalg.norm(values)
    if norm > 0:
        values /= norm
    return unique_bins, values


def fragment_similarity(query_bins, query_values, ref_bins, ref_values):
    """Sparse cosine similarity between two binned spectra."""
    if len(query_bins) == 0 or len(ref_bins) == 0:
        return 0.0
    _, query_idx, ref_idx = np.intersect1d(
        query_bins, ref_bins, assume_unique=True, return_indices=True
    )
    if len(query_idx) == 0:
        return 0.0
    return float(np.dot(query_values[query_idx], ref_values[ref_idx]))


def get_candidate_positions(adduct, precursor_mz):
    """Retrieve nearby precursor references; fall back to nearest masses."""
    if not np.isfinite(precursor_mz):
        precursor_mz = float(train["precursor_mz"].median())

    entry = adduct_index.get(str(adduct))
    positions = np.empty(0, dtype=np.int64)
    masses = None
    if entry is not None:
        masses, adduct_positions = entry
        for ppm in PRECURSOR_WINDOWS_PPM:
            delta = max(abs(float(precursor_mz)) * ppm / 1_000_000, 0.002)
            left = np.searchsorted(masses, precursor_mz - delta, side="left")
            right = np.searchsorted(masses, precursor_mz + delta, side="right")
            if right > left:
                positions = adduct_positions[left:right]
                break

    if len(positions) == 0:
        # Rare out-of-domain case: nearest precursor masses, preferring the
        # same adduct when available.
        if entry is not None and len(masses):
            nearest = np.searchsorted(masses, precursor_mz)
            lo = max(0, nearest - MAX_CANDIDATE_SPECTRA // 2)
            hi = min(len(masses), lo + MAX_CANDIDATE_SPECTRA)
            positions = entry[1][lo:hi]
        else:
            nearest = np.searchsorted(all_masses, precursor_mz)
            lo = max(0, nearest - MAX_CANDIDATE_SPECTRA // 2)
            hi = min(len(all_masses), lo + MAX_CANDIDATE_SPECTRA)
            positions = all_positions[lo:hi]

    if len(positions) > MAX_CANDIDATE_SPECTRA:
        ref_masses = train.loc[positions, "precursor_mz"].to_numpy(dtype=np.float64)
        keep = np.argpartition(
            np.abs(ref_masses - precursor_mz), MAX_CANDIDATE_SPECTRA - 1
        )[:MAX_CANDIDATE_SPECTRA]
        positions = positions[keep]
    return positions


# Smoke-test the matching helpers without running the full inference loop.
_demo_bins, _demo_values = spectrum_vector([50.0, 75.0], [1.0, 0.5])
assert fragment_similarity(_demo_bins, _demo_values, _demo_bins, _demo_values) > 0.99
assert len(get_candidate_positions(test.iloc[0]["adduct"], test.iloc[0]["precursor_mz"])) > 0
print("Index ready:", len(adduct_index), "adducts; sparse-fragment smoke test passed.")

In [ ]:
# Load the public CASMI spectrum-to-fingerprint model and CC-BY candidate library.
# Checkpoint layer names match the published FPNet state dictionary.

import glob
import math
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F


def find_kaggle_input(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Missing {filename}; verify attached Kaggle inputs.")
    return sorted(matches, key=len)[0]


class SinEmb(nn.Module):
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)
        self.register_buffer("inv", (2 * math.pi) / wav)

    def forward(self, x):
        angles = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(angles), torch.cos(angles)], -1)


class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__()
        self.h = h
        self.n1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.o = nn.Linear(d, d)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(drop)

    def forward(self, x, pad):
        batch, n, dim = x.shape
        y = self.n1(x)
        q, k, v = self.qkv(y).view(batch, n, 3, self.h, dim // self.h).permute(2, 0, 3, 1, 4)
        attention = F.scaled_dot_product_attention(q, k, v, attn_mask=(~pad)[:, None, None, :])
        x = x + self.drop(self.o(attention.transpose(1, 2).reshape(batch, n, dim)))
        return x + self.drop(self.ff(self.n2(x)))


class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d = d
        self.mz_emb = SinEmb(d)
        self.nl_emb = SinEmb(d)
        self.pk = nn.Linear(2 * d + 1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(26, d)
        self.ins = nn.Embedding(5, d)
        self.gl = nn.Linear(d + 3, d)
        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))

    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        batch, _ = mz.shape
        neutral_loss = (prec[:, None] - mz).clamp(min=0)
        peaks = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(neutral_loss), it.unsqueeze(-1)], -1))
        global_token = self.gl(torch.cat([
            self.prec_emb(prec), (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),
            torch.log1p(prec).unsqueeze(-1) / 10.0,
        ], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([global_token.unsqueeze(1), peaks], 1)
        pad = torch.cat([torch.zeros(batch, 1, dtype=torch.bool, device=pad.device), pad], 1)
        for block in self.blocks:
            x = block(x, pad)
        x = self.norm(x)
        valid = (~pad[:, 1:]).float().unsqueeze(-1)
        mean = (x[:, 1:] * valid).sum(1) / valid.sum(1).clamp(min=1)
        return self.head(torch.cat([x[:, 0], mean], -1))


FP_ADDUCTS = [
    "[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",
    "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",
    "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",
    "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>",
]
FP_ADDUCT_INDEX = {name: i for i, name in enumerate(FP_ADDUCTS)}


def instrument_family(value):
    text = str(value).lower()
    if "timstof" in text:
        return 0
    if any(token in text for token in ("orbitrap", "qft", "ftms", "hybrid ft", "itft", "exactive")):
        return 1
    if "tof" in text:
        return 2
    if "trap" in text or "qq" in text:
        return 3
    return 4


def prepare_fp_peaks(mzs, intensities, precursor, max_peaks=128):
    mzs = np.asarray(mzs, dtype=np.float64).reshape(-1)
    intensities = np.asarray(intensities, dtype=np.float64).reshape(-1)
    if len(mzs) == 0 or len(mzs) != len(intensities):
        return np.zeros(0, np.float32), np.zeros(0, np.float32)
    valid = np.isfinite(mzs) & np.isfinite(intensities) & (mzs <= precursor + 1.5)
    mzs, intensities = mzs[valid], intensities[valid]
    if len(mzs) == 0 or np.max(intensities) <= 0:
        return np.zeros(0, np.float32), np.zeros(0, np.float32)
    valid = intensities >= 1e-3 * intensities.max()
    mzs, intensities = mzs[valid], intensities[valid]
    if len(mzs) > max_peaks:
        order = np.argsort(-intensities)
        selected, bucket_counts = [], {}
        for index in order:
            bucket = int(mzs[index] // 50.0)
            if bucket_counts.get(bucket, 0) < 8:
                bucket_counts[bucket] = bucket_counts.get(bucket, 0) + 1
                selected.append(index)
                if len(selected) == max_peaks:
                    break
        if len(selected) < max_peaks:
            selected_set = set(selected)
            selected.extend([i for i in order if i not in selected_set][:max_peaks - len(selected)])
        mzs, intensities = mzs[selected], intensities[selected]
    order = np.argsort(mzs)
    mzs, intensities = mzs[order], intensities[order]
    return mzs.astype(np.float32), np.sqrt(intensities / intensities.max()).astype(np.float32)


def load_fp_checkpoint(path, device):
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = FPNet(checkpoint["nbits"], d=checkpoint["d"], layers=checkpoint["layers"])
    model.load_state_dict(checkpoint["model"])
    return model.to(device).eval(), int(checkpoint["nbits"])


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FP_SINGLE_PATH = find_kaggle_input("fp_single_s2.pt")
FP_MERGED_PATH = find_kaggle_input("fp_merged_m1.pt")
COCO_FP_PATH = find_kaggle_input("coco_fp.npy")
COCO_MASS_PATH = find_kaggle_input("coco_mass.npy")
COCO_META_PATH = find_kaggle_input("coco_meta.pkl")
FP_BITS_PATH = find_kaggle_input("fp_bits.npy")
fp_single, fp_nbits = load_fp_checkpoint(FP_SINGLE_PATH, DEVICE)
fp_merged, merged_nbits = load_fp_checkpoint(FP_MERGED_PATH, DEVICE)
if fp_nbits != merged_nbits:
    raise ValueError("Fingerprint checkpoints have different output sizes.")
fp_nbits = min(fp_nbits, len(np.load(FP_BITS_PATH)))
coco_fp_packed = np.load(COCO_FP_PATH, mmap_mode="r")
coco_mass = np.asarray(np.load(COCO_MASS_PATH, mmap_mode="r"), dtype=np.float64)
with open(COCO_META_PATH, "rb") as handle:
    coco_meta = pickle.load(handle)
coco_keys = np.asarray(coco_meta["keys"], dtype=object)
coco_smiles = np.asarray(coco_meta["smiles"], dtype=object)
if not (len(coco_mass) == len(coco_keys) == len(coco_smiles) == len(coco_fp_packed)):
    raise ValueError("COCONUT fingerprint, mass and metadata rows are not aligned.")
coco_order = np.argsort(coco_mass)
coco_mass_sorted = coco_mass[coco_order]
coco_index_sorted = coco_order
FP_BATCH_SIZE = 32
print(f"Pretrained FPNet ready on {DEVICE}; {fp_nbits:,} bits; {len(coco_mass):,} candidates.")

In [ ]:
# Hybrid FPNet + spectral-library inference; write and validate submission.csv.

from collections import defaultdict


def row_mode(row):
    value = str(row.get("ionization_mode", "")).lower()
    if value in ("positive", "pos", "+", "1", "true"):
        return 1.0
    if value in ("negative", "neg", "-", "-1", "false"):
        return -1.0
    return 1.0 if str(row.get("adduct", "")).endswith("+") else -1.0


def row_collision_energy(row):
    for column in ("collision_energy_ev", "collision_energy_orig_units"):
        if column in row.index:
            try:
                value = float(row[column])
                if np.isfinite(value):
                    return value
            except (TypeError, ValueError):
                pass
    return 25.0


def batch_fp_logits(model, prepared, precursors, adducts, instruments, energies, modes):
    outputs = []
    for start in range(0, len(prepared), FP_BATCH_SIZE):
        end = min(start + FP_BATCH_SIZE, len(prepared))
        spectra = prepared[start:end]
        width = max(1, max((len(x[0]) for x in spectra), default=0))
        mz = np.zeros((len(spectra), width), dtype=np.float32)
        intensity = np.zeros((len(spectra), width), dtype=np.float32)
        pad = np.ones((len(spectra), width), dtype=bool)
        for i, (peak_mz, peak_intensity) in enumerate(spectra):
            count = len(peak_mz)
            if count:
                mz[i, :count] = peak_mz
                intensity[i, :count] = peak_intensity
                pad[i, :count] = False
        with torch.inference_mode():
            logits = model(
                torch.as_tensor(mz, device=DEVICE),
                torch.as_tensor(intensity, device=DEVICE),
                torch.as_tensor(pad, device=DEVICE),
                torch.as_tensor(precursors[start:end], dtype=torch.float32, device=DEVICE),
                torch.as_tensor(
                    [FP_ADDUCT_INDEX.get(str(a), FP_ADDUCT_INDEX["<unk>"]) for a in adducts[start:end]],
                    dtype=torch.long, device=DEVICE,
                ),
                torch.as_tensor(instruments[start:end], dtype=torch.long, device=DEVICE),
                torch.as_tensor(energies[start:end], dtype=torch.float32, device=DEVICE),
                torch.as_tensor(modes[start:end], dtype=torch.float32, device=DEVICE),
            )
        outputs.append(logits.float().cpu().numpy())
    return np.concatenate(outputs, axis=0) if outputs else np.zeros((0, fp_nbits), dtype=np.float32)


# Convert ion mass to neutral molecular mass for matching against COCONUT.
PROTON = 1.0072764666
ADDUCT_NEUTRAL_SHIFT = {
    "[M+H]+": (1, PROTON), "[M+NH4]+": (1, 18.033823), "[M+Na]+": (1, 22.989218),
    "[M+K]+": (1, 38.963158), "[M-H2O+H]+": (1, -17.003288),
    "[M-2H2O+H]+": (1, -35.013852), "[M-H]-": (1, -PROTON),
    "[M-H2O-H]-": (1, -19.017841), "[M+CH2O2-H]-": (1, 44.998201),
    "[M+C2H4O2-H]-": (1, 59.013851), "[M+Cl]-": (1, 34.969402),
    "[M]+": (1, 0.0), "[M-H2O]+": (1, -18.010565), "[M+2H]2+": (2, 2 * PROTON),
    "[M-2H]-": (2, -2 * PROTON), "[M+Na-2H]-": (1, 20.974666),
    "[2M+H]+": (1, PROTON), "[2M+Na]+": (1, 22.989218), "[2M+NH4]+": (1, 18.033823),
    "[2M+K]+": (1, 38.963158), "[2M-H]-": (1, -PROTON),
}
ADDUCT_MULTIPLIER = {"[2M+H]+": 2, "[2M+Na]+": 2, "[2M+NH4]+": 2,
                     "[2M+K]+": 2, "[2M-H]-": 2, "[3M+H]+": 3, "[3M-H]-": 3}


def neutral_precursor(precursor_mz, adduct):
    adduct = str(adduct)
    charge, shift = ADDUCT_NEUTRAL_SHIFT.get(
        adduct, (1, PROTON if adduct.endswith("+") else -PROTON)
    )
    multiplier = ADDUCT_MULTIPLIER.get(adduct, 1)
    return (float(precursor_mz) * charge - shift) / multiplier


# First build direct reference-library evidence. Keep this dictionary separate
# from FPNet candidate scores so the two evidence channels are not overwritten.
train_molecules = train["inchikey14"].to_numpy(dtype=object)
train_smiles = train["normalized_smiles"].to_numpy(dtype=object)
train_ms2_mzs = train["ms2_mzs"].to_numpy(dtype=object)
train_ms2_intensities = train["ms2_normalized_intensities"].to_numpy(dtype=object)
train_precursors = train["precursor_mz"].to_numpy(dtype=np.float64)

smiles_lookup = (
    train.groupby(["inchikey14", "normalized_smiles"], sort=False).size()
    .rename("count").reset_index()
    .sort_values(["inchikey14", "count"], ascending=[True, False])
    .drop_duplicates("inchikey14").set_index("inchikey14")["normalized_smiles"]
    .astype(str).to_dict()
)
frequent_molecules = train["inchikey14"].value_counts().index.astype(str).tolist()

from functools import lru_cache

@lru_cache(maxsize=50000)
def reference_vector(position):
    position = int(position)
    return spectrum_vector(train_ms2_mzs[position], train_ms2_intensities[position])

library_scores = defaultdict(dict)
for row_number, query in test.iterrows():
    molecule_id = str(query["molecule_id"])
    precursor = float(query["precursor_mz"])
    if not np.isfinite(precursor):
        precursor = float(train["precursor_mz"].median())
    positions = get_candidate_positions(query["adduct"], precursor)
    q_bins, q_values = spectrum_vector(query["ms2_mzs"], query["ms2_normalized_intensities"])
    for position in positions:
        position = int(position)
        r_bins, r_values = reference_vector(position)
        similarity = fragment_similarity(q_bins, q_values, r_bins, r_values)
        ppm_error = abs(train_precursors[position] - precursor) / max(abs(precursor), 1e-12) * 1e6
        score = similarity + 0.02 * np.exp(-ppm_error / 10.0)
        key = str(train_molecules[position])
        if score > library_scores[molecule_id].get(key, (-np.inf, ""))[0]:
            library_scores[molecule_id][key] = (float(score), smiles_lookup.get(key, str(train_smiles[position])))
    if (row_number + 1) % 200 == 0 or row_number + 1 == len(test):
        print(f"Library spectra scored: {row_number + 1:,}/{len(test):,}")

# Run single-spectrum model over all test spectra.
prepared_test, test_precursors, test_adducts = [], [], []
test_instruments, test_energies, test_modes, test_neutral_masses = [], [], [], []
for _, row in test.iterrows():
    precursor = float(row["precursor_mz"])
    if not np.isfinite(precursor):
        precursor = float(train["precursor_mz"].median())
    adduct = str(row["adduct"])
    prepared_test.append(prepare_fp_peaks(row["ms2_mzs"], row["ms2_normalized_intensities"], precursor))
    test_precursors.append(precursor)
    test_adducts.append(adduct)
    test_instruments.append(instrument_family(row.get("instrument_type", "other")))
    test_energies.append(row_collision_energy(row))
    test_modes.append(row_mode(row))
    test_neutral_masses.append(neutral_precursor(precursor, adduct))

test_precursors = np.asarray(test_precursors, dtype=np.float32)
test_instruments = np.asarray(test_instruments, dtype=np.int64)
test_energies = np.asarray(test_energies, dtype=np.float32)
test_modes = np.asarray(test_modes, dtype=np.float32)
test_neutral_masses = np.asarray(test_neutral_masses, dtype=np.float64)
print(f"Predicting fingerprints on {DEVICE}...")
single_logits = batch_fp_logits(
    fp_single, prepared_test, test_precursors, test_adducts,
    test_instruments, test_energies, test_modes,
)

# Prepare merged spectra per molecule for the separately trained merged model.
molecule_to_rows = defaultdict(list)
for position, molecule_id in enumerate(test["molecule_id"].astype(str)):
    molecule_to_rows[molecule_id].append(position)


def merge_raw_spectra(rows):
    peak_mz_parts, peak_int_parts = [], []
    for _, row in rows.iterrows():
        mz = np.asarray(row["ms2_mzs"], dtype=np.float32)
        intensities = np.asarray(row["ms2_normalized_intensities"], dtype=np.float32)
        if len(mz) == len(intensities) and len(mz):
            peak_mz_parts.append(mz)
            peak_int_parts.append(intensities / max(float(np.max(intensities)), 1e-9))
    if not peak_mz_parts:
        return np.zeros(0, np.float32), np.zeros(0, np.float32)
    mz = np.concatenate(peak_mz_parts)
    intensities = np.concatenate(peak_int_parts)
    order = np.argsort(mz)
    mz, intensities = mz[order], intensities[order]
    keep = np.ones(len(mz), dtype=bool)
    for i in range(1, len(mz)):
        if mz[i] - mz[i - 1] < 0.005:
            if intensities[i] >= intensities[i - 1]:
                keep[i - 1] = False
            else:
                keep[i] = False
    return mz[keep], intensities[keep]


merged_prepared, merged_precursors, merged_adducts = [], [], []
merged_instruments, merged_energies, merged_modes, merged_ids = [], [], [], []
for molecule_id, positions in molecule_to_rows.items():
    rows = test.iloc[positions]
    precursor = float(np.median(test_precursors[positions]))
    raw_mz, raw_intensity = merge_raw_spectra(rows)
    if not len(raw_mz):
        continue
    merged_prepared.append(prepare_fp_peaks(raw_mz, raw_intensity, precursor))
    merged_precursors.append(precursor)
    merged_adducts.append(test_adducts[positions[0]])
    merged_instruments.append(int(test_instruments[positions[0]]))
    merged_energies.append(25.0)
    merged_modes.append(float(np.mean(test_modes[positions])))
    merged_ids.append(molecule_id)

merged_logits = batch_fp_logits(
    fp_merged, merged_prepared,
    np.asarray(merged_precursors, dtype=np.float32), merged_adducts,
    np.asarray(merged_instruments, dtype=np.int64),
    np.asarray(merged_energies, dtype=np.float32),
    np.asarray(merged_modes, dtype=np.float32),
)
merged_position = {molecule_id: i for i, molecule_id in enumerate(merged_ids)}
molecule_logits = {}
for molecule_id, positions in molecule_to_rows.items():
    logits = single_logits[positions].mean(axis=0)
    if molecule_id in merged_position:
        logits = 0.5 * (logits + merged_logits[merged_position[molecule_id]])
    molecule_logits[molecule_id] = logits.astype(np.float32)

# Rank only the COCONUT mass-window candidates for each test molecule.
fp_scores_by_molecule = {}
for molecule_id, positions in molecule_to_rows.items():
    candidate_indices = set()
    for position in positions:
        target_mass = test_neutral_masses[position]
        if not np.isfinite(target_mass) or target_mass <= 0:
            continue
        for ppm in (10.0, 25.0, 50.0):
            delta = max(target_mass * ppm / 1_000_000.0, 0.003)
            left = np.searchsorted(coco_mass_sorted, target_mass - delta, side="left")
            right = np.searchsorted(coco_mass_sorted, target_mass + delta, side="right")
            if right > left:
                candidate_indices.update(coco_index_sorted[left:right].tolist())
                break
    candidate_indices = np.asarray(sorted(candidate_indices), dtype=np.int64)
    scores_for_molecule = {}
    if len(candidate_indices):
        fingerprints = np.unpackbits(np.asarray(coco_fp_packed[candidate_indices]), axis=1)
        fingerprints = fingerprints[:, :fp_nbits].astype(np.float32, copy=False)
        scores = fingerprints @ molecule_logits[molecule_id]
        for candidate_index, score in zip(candidate_indices, scores):
            key = str(coco_keys[candidate_index])
            previous = scores_for_molecule.get(key)
            if previous is None or float(score) > previous[0]:
                scores_for_molecule[key] = (float(score), str(coco_smiles[candidate_index]))
    fp_scores_by_molecule[molecule_id] = scores_for_molecule

# Blend evidence by within-molecule rank: FPNet supplies unseen natural-product
# candidates, while direct spectrum matches contribute a 25% rank bonus.
submission_rows = []
for molecule_id in sample_submission["molecule_id"].astype(str):
    fp_candidates = fp_scores_by_molecule.get(molecule_id, {})
    library_candidates = library_scores.get(molecule_id, {})
    fp_order = sorted(fp_candidates, key=lambda key: fp_candidates[key][0], reverse=True)
    fp_rank = {key: (len(fp_order) - i) / max(len(fp_order), 1) for i, key in enumerate(fp_order)}
    library_order = sorted(library_candidates, key=lambda key: library_candidates[key][0], reverse=True)
    library_rank = {key: (len(library_order) - i) / max(len(library_order), 1) for i, key in enumerate(library_order)}

    all_keys = set(fp_candidates) | set(library_candidates)
    smiles_for_key = {}
    combined_score = {}
    for key in all_keys:
        if key in fp_candidates:
            smiles_for_key[key] = fp_candidates[key][1]
        if key in library_candidates:
            smiles_for_key[key] = library_candidates[key][1]
        combined_score[key] = fp_rank.get(key, 0.0) + 0.25 * library_rank.get(key, 0.0)

    guesses, seen_smiles = [], set()
    for key in sorted(all_keys, key=lambda item: combined_score[item], reverse=True):
        smiles = smiles_for_key.get(key)
        if smiles and smiles not in seen_smiles:
            guesses.append(smiles)
            seen_smiles.add(smiles)
        if len(guesses) == MAX_GUESSES:
            break
    if len(guesses) < MAX_GUESSES:
        for key in frequent_molecules:
            smiles = smiles_lookup.get(key)
            if smiles and smiles not in seen_smiles:
                guesses.append(smiles)
                seen_smiles.add(smiles)
            if len(guesses) == MAX_GUESSES:
                break
    if not guesses:
        raise ValueError(f"No valid SMILES candidates for molecule_id={molecule_id}")
    submission_rows.append({"molecule_id": molecule_id, "smiles": ";".join(guesses)})

submission = pd.DataFrame(submission_rows, columns=["molecule_id", "smiles"])
output_path = Path("/kaggle/working/submission.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)
candidate_counts = submission["smiles"].str.split(";").map(len)
assert submission["molecule_id"].is_unique
assert submission["molecule_id"].tolist() == sample_submission["molecule_id"].astype(str).tolist()
assert submission["smiles"].notna().all() and submission["smiles"].str.len().gt(0).all()
assert candidate_counts.between(1, MAX_GUESSES).all()
print(f"Saved {output_path}: {len(submission):,} rows; {candidate_counts.min()}–{candidate_counts.max()} guesses/row.")
print(submission.head(3).to_string(index=False))